In [ ]:
import cv2
import numpy as np
from astropy.io import fits
from astroquery.gaia import Gaia
from astropy.coordinates.sky_coordinate import SkyCoord
from astropy.wcs import WCS
import sep
sep.set_extract_pixstack(1000000)
sep.set_sub_object_limit(1000000)

from astropy.table import Table, hstack
from scipy.optimize import curve_fit
import csv
import math 
from scipy.stats import mode as spmode

import matplotlib.pyplot as plt
from astropy.visualization import SqrtStretch
from astropy.visualization.mpl_normalize import ImageNormalize
from astropy.stats import SigmaClip
from photutils.background import Background2D, MedianBackground
from photutils.background import *
from photutils.datasets import make_100gaussians_image
from astropy.stats import biweight_location
from astropy.stats import mad_std
from astropy.stats import sigma_clipped_stats

import astropy.table
from matplotlib.backends.backend_pdf import PdfPages

import numpy as np
from astropy.wcs import WCS
from astropy.table import Table, hstack, vstack

import os
from astropy.io import ascii
import glob
from glob import glob

from astroquery.astrometry_net import AstrometryNet

from datetime import datetime
from astropy.time import Time
import matplotlib.dates as mdates

# sq func used in analysis 
def sq(x,a,b):
    return(a * (np.sign(x) * (np.abs(x))) **(b/2) )
    
import pandas as pd

mean = np.mean
std = np.std


# for handling arrays; image data are arrays!
import numpy as np 
# for plotting and displaying data
import matplotlib.pyplot as plt 
import matplotlib.gridspec as gridspec
# for making lists of filenames
from glob import glob 
# for reading and writing FITS format data
from astropy.io import fits 
# for doing aperture photometry:
import photutils
from photutils.detection import DAOStarFinder
from photutils.aperture import CircularAperture, CircularAnnulus
from astropy.stats import mad_std
# Import a function for fitting data
from scipy.optimize import curve_fit
# need to manually trigger garbage collector to free memory
import gc
from IPython.display import display, clear_output
import scipy.optimize as optimize


############### Set the platescale for your observation ##################

#platescale = 0.36 # arcsec / pixel    #ProEM Camera on the McDonald 82" telescope
#platescale = 0.964016 # arcsec / pixel   #CMOS on the Yerkes 41" telescope
platescale = 0.6 # arcsec / pixel    #CCD on the Yekres 24" telescope

#platescale=        # place to put platescales for other telescopes

In [ ]:
'''def phot2(data, header):
    bkg = sep.Background(data,bw=125, bh=125, fw=11, fh=11)
    data_sub = data - bkg
    objects = Table(sep.extract(data_sub, 1.5, err = bkg.globalrms))
    objects['flux2'], fluxerr, flag = sep.sum_circle(data_sub, objects['x'], objects['y'],
                                     3.0, err=bkg.globalrms)



    # evaluate background as 2-d array, same size as original image
    bkg_image = bkg.back()
    # bkg_image = np.array(bkg) # equivalent to above
    # show the background
    plt.figure()
    plt.imshow(bkg_image, interpolation='nearest', cmap='gray', origin='lower')
    plt.colorbar()
    plt.show()
    
    
    
    # evaluate the background noise as 2-d array, same size as original image
    bkg_rms = bkg.rms()
    
    plt.figure()
    # show the background noise
    plt.imshow(bkg_rms, interpolation='nearest', cmap='gray', origin='lower')
    plt.colorbar()
    plt.show()
    
    return objects, bkg






def phot3(data, header, sigma, mbox_num, fsize):

    sigma_clip = SigmaClip(sigma = sigma)
    bkg_estimator = MMMBackground()
    # background detection
    bkg = Background2D(data, int(data.shape[0]/mbox_num),  filter_size=(fsize, fsize), sigma_clip=sigma_clip, bkg_estimator = bkg_estimator)
    # background subtraction
    data_sub = data - bkg.background




    
    # Extract objects/stars from the image
    objects = Table(sep.extract(data_sub, thresh=1.5, err=bkg.globalrms))
    
    # 1. Automatically determine a half-light/core radius (e.g., enclosing 50% of light)
    # rmax defines the maximum search radius boundary in pixels
    rmax = 10.0
    half_light_rad = sep.flux_radius(data_sub, objects['x'], objects['y'], rmax, 0.5)
    
    # 2. Scale radii automatically for the circular annulus (e.g., sky annulus from 2x to 3x of core radius)
    rin = 2.0 * half_light_rad
    rout = 3.0 * half_light_rad
    
    # 3. Measure flux/sum within the automatically determined circular annulus
    annulus_sum, annulus_err, annulus_flags = sep.sum_circann(
        data_sub, objects['x'], objects['y'], rin, rout)'''

In [ ]:


def phot(data, header, sigma, mbox_num, fsize):

    sigma_clip = SigmaClip(sigma = sigma)
    bkg_estimator = MMMBackground()
    # background detection
    bkg = Background2D(data, int(data.shape[0]/mbox_num),  filter_size=(fsize, fsize), sigma_clip=sigma_clip, bkg_estimator = bkg_estimator)
    # background subtraction
    data_sub = data - bkg.background

    #source extraction
    objects = Table(sep.extract(data_sub, sigma, err = bkg.background_rms_median))
    ob_mask = np.where(objects['flag'] != 8)[0]
    objects = objects[ob_mask]
    print(len(objects))

    # measure flux
    #objects['flux2'], sum_err, flag = sep.sum_ellipse(data=data_sub, x=objects['x'], y=objects['y'], a=objects['a'], b=objects['b'], theta=objects['theta'],err=bkg.background_rms_median)
    objects['flux2'], objects['flux2_err'], flag = sep.sum_circle(data=data_sub, x=objects['x'], y=objects['y'], r = ((objects['xmax'] -  objects['xmin'])/2))
    #objects['flux2'], objects['flux2_err'], flag = sep.sum_circann(data=data_sub, x=objects['x'], y=objects['y'], rin = (objects['xmax'] -  objects['xmin'])/2, rout = (((objects['xmax'] -  objects['xmin'])/2)+5)                                               )
    
    return objects, bkg 



def gaia(header):
    wcs_header = WCS(header)
    bottom_coord = wcs_header.pixel_to_world(0, 0)
    top_coord = wcs_header.pixel_to_world(header['NAXIS1'], header['NAXIS2'])
    full_coord_ra = np.abs(top_coord.ra.deg-bottom_coord.ra.deg)
    full_coord_dec = np.abs(top_coord.dec.deg-bottom_coord.dec.deg)
    ra_center = (top_coord.ra.deg + bottom_coord.ra.deg) / 2
    dec_center = (top_coord.dec.deg + bottom_coord.dec.deg) / 2
    job = Gaia.launch_job("SELECT TOP 300000 "
                    "source_id,ra,dec,parallax,parallax_error,pm,pmra,pmra_error,pmdec,pmdec_error,"
                    "phot_g_mean_mag, phot_g_mean_flux, phot_bp_mean_mag,phot_bp_mean_flux,phot_rp_mean_mag,"
                    "phot_rp_mean_flux,bp_rp, phot_variable_flag, classprob_dsc_combmod_galaxy"
                    " from gaiadr3.gaia_source"
                    " WHERE CONTAINS(POINT('ICRS',ra,dec),BOX('ICRS',{0},{1},{2},{3}))=1 AND"
                    "(phot_bp_mean_mag <= 18.0)".format(ra_center,
                                                                      dec_center,
                                                                    full_coord_ra, full_coord_dec))
    gaia_res = job.get_results()
    gaia_coord = SkyCoord(gaia_res['ra'], gaia_res['dec'], frame = 'icrs', unit = 'deg')
    
    return gaia_res, gaia_coord




def match_table(object_table, wcs, gaia_table, gaia_coordinates):
    pcor = wcs.pixel_to_world(object_table['x'], object_table['y'])
    object_table['ra_p'], object_table['dec_p'] = pcor.ra.deg, pcor.dec.deg
    index, d2d, ____ = pcor.match_to_catalog_sky(gaia_coordinates)
    mtab = hstack([gaia_table[index], Table(object_table)])
    _, unique_ind = np.unique(mtab['source_id'], return_index=True)
    mtab = mtab[unique_ind]
    mtab['ang_dist'] = np.abs(np.sqrt(mtab['ra']**2+mtab['dec']**2)
                              -np.sqrt(mtab['ra_p']**2+mtab['dec_p']**2))
    mtab['dec_res'] = (mtab['dec'] -  mtab['dec_p'])*3600
    mtab['ra_res'] = (mtab['ra'] -  mtab['ra_p'])*3600*0.9114
        
    return mtab



def zp(match, header, flux_name):
    zp_list = []
    exp = header['EXPTIME']
    match2 = match
    # if you want to filter out what stars it considers
    #tfilter = np.where(match['phot_bp_mean_mag'] <= 18) & (match['phot_variable_flag'] != 'VARIABLE')[0]
    #tfilter = np.where(match['phot_bp_mean_mag'] < 13) & (match['phot_bp_mean_mag'] >= 12)[0]
    #match2 = match[tfilter]
   
    
    for m in range(len(match2)):
        gaia_mag = match2['phot_bp_mean_mag'][m]
        flux = match2[flux_name][m]
        zp = gaia_mag + 2.5*np.log10(flux/exp) # main equation
        zp_list.append(zp)

    avg_zp = np.nanmean(zp_list)
    return avg_zp



'''def zp(match, header, flux_name):
    
    exp = header['EXPTIME']

    match['zp'] = match['phot_bp_mean_mag'] + 2.5*np.log10(match[flux_name]/exp)
        
    return match
'''


def mag_comp(match, avg_zp, header, flux_name):
    exp = header['EXPTIME']
    match['new_mag'] = -2.5 * np.log10(match[flux_name]/exp) + avg_zp # main equation

    return match
        
def display(dat):
    plt.figure() #Create new "figure" for this image
    plt.imshow(dat,cmap='gray',vmin=np.percentile(data,5),
           vmax=np.percentile(data,98), origin = 'lower')
    plt.show()



def bkg_plots(data, bkg):
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3)
    fig.set_figheight(8)
    fig.set_figwidth(15)
    p1 = ax1.imshow(bkg.background, origin='lower', cmap='Greys_r', interpolation='nearest')
    plt.colorbar(p1, ax = ax1, shrink=0.4)
    ax1.set_title('Bkg')                                           
    p2 = ax2.imshow(bkg.background_rms, origin='lower', cmap='Greys_r', interpolation='nearest')
    plt.colorbar(p2, ax = ax2, shrink=0.4)
    ax2.set_title('RMS\n Med: %.4f   STD: %.4f'%(bkg.background_rms_median, std(bkg.background_rms)))
    ax3.imshow(data,cmap='gray',vmin=np.percentile(data,5),vmax=np.percentile(data,98), origin = 'lower')
    bkg.plot_meshes(outlines=True, marker='.', color='cyan', alpha=0.3)
    plt.show()
        
    
    

In [ ]:
%matplotlib inline
path = '/raid6/users/rantoine/2026-07-13/lights/7-13-26/7-13-26_c/'
file = '7-13-26_R_Aql*_c.new'
outdir = '/raid6/users/rantoine/Mira_Notebooks/Better_Part_Three/mtab2_output/'
os.makedirs(outdir, exist_ok=True)

files = sorted(glob(path + file))
print(len(files))
print(files)
header

In [ ]:

%matplotlib inline
with fits.open(files[0]) as hdu:
    header = hdu[0].header

# gaia query
gtab, gcor = gaia(header)



for f in range(len((files))):
    file = files[f]
    print('analyzing ' + str(file))
    with fits.open(file) as hdu:
        data = hdu[0].data.astype(np.float32)

    display(data)

    wcs= WCS(header)

    # background subtraction and photometry
    otab, bkg = phot(data, header, sigma = 3, mbox_num = 15, fsize = 11)


    bkg_plots(data, bkg)
    display(dat = data-bkg.background)


    # match detected sources and gaia sources
    mtab = match_table(otab, wcs, gtab,gcor)

    # calculate an average zero point magnitude for all sources
    avg_zp = zp(mtab, header, flux_name = 'flux2')
    print(avg_zp)

    # calculate magnitudes with zero point
    mtab2 = mag_comp(mtab, avg_zp, header, flux_name = 'flux2')

    index = np.where(mtab2['source_id'] == 4307450193249412352)[0]
    print(index)
    print(mtab2['new_mag'][index])

    plt.figure()
    plt.scatter(mtab2['new_mag'], mtab2['phot_bp_mean_mag'], alpha = 0.4)
    plt.scatter(mtab2['new_mag'][index], mtab2['phot_bp_mean_mag'][index], color = 'r', alpha = 0.4)
    #plt.scatter(mtab3['new_mag'], mtab3['phot_bp_mean_mag'], alpha = 0.4)
    #plt.plot(x_fit, y_fit, label=f'Fit: a={popt[0]:.2f}, b={popt[1]:.2f}')
    plt.xlabel('Calibrated Mag')
    plt.ylabel('Gaia mag')
    #plt.xlim(7,20)
    #plt.ylim(7,20)
    plt.show()
    plt.figure() #Create new "figure" for this image
    plt.imshow(data,cmap='gray',vmin=np.percentile(data,5),vmax=np.percentile(data,98), origin = 'lower')


#plt.hexbin(mtab2['x'], mtab2['y'], C=mtab2['bp_rp'], gridsize = 50 )

    plt.hexbin(mtab2['x'], mtab2['y'], C=mtab2['phot_bp_mean_mag']-mtab2['new_mag'], gridsize = 100)
    plt.colorbar(label = 'gaia bp mag - cal mag')
    plt.show()
    mtab2['phot_variable_flag'] = mtab2['phot_variable_flag'].astype(str)
    outname = os.path.join(outdir, os.path.basename(filepath).replace('.new', 'test_mtab2.fits'))
    mtab2.write(outname, overwrite=True)


In [ ]:
######### CHANGE FILE NAME #########
mtab2_files = sorted(glob(os.path.join(outdir, '7-13-26*test_mtab2.fits')))
print(len(mtab2_files))
print(mtab2_files)

In [ ]:

r_aql_rows = []
standard_rows = []

for savedfile in mtab2_files:
    mtab2 = Table.read(savedfile)

    index = np.where(mtab2['source_id'] == 4307450193249412352)[0]
    index2 = np.where(mtab2['source_id'] == 4307450468131938432)[0]
    
    r_aql = mtab2[index]
    standard = mtab2[index2]
    
    r_aql_rows.append(r_aql)
    standard_rows.append(standard)

    r_aql_all = vstack(r_aql_rows)
    standard_all = vstack(standard_rows)

    ######## change file name ######### 
    
    r_aql_all.write('2026-07-13_R_Aql_test_mtab2.csv', format='csv', overwrite=True)
    standard_all.write('2026-07-13_standard_test_mtab2.csv', format='csv', overwrite=True)

In [ ]:
plt.close()
obs_num = np.arange(len(r_aql_all))

plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
    'figure.titlesize': 18
})

fig, ax = plt.subplots(figsize=(10, 5))

ax.scatter(obs_num, r_aql_all['new_mag'], color='#9C179E', s=10)
ax.scatter(obs_num, standard_all['new_mag'], color = 'b', s=10)
ax.set_xlabel('Observation')
ax.set_ylabel('Magnitude (B)')
ax.invert_yaxis()
ax.set_title('Yerkes Light Curve of R Aql and Standard Star')



fig.autofmt_xdate()
plt.show()

In [ ]:
r_aql_mag_files = sorted(glob('/raid6/users/rantoine/Mira_Notebooks/Better_Part_Three/*R_Aql_mtab2.csv'))

r_aql_dates = []
r_aql_avg_mag = []
for filepath in r_aql_mag_files:
    filename = os.path.basename(filepath)
    date_str = filename.split('_R_Aql')[0] #taking the date from the filename
    date_time = datetime.strptime(date_str, "%Y-%m-%d")
    #date = Time(date_time, scale="utc")
    
    df = pd.read_csv(filepath)
    mag = df['new_mag']
    mag = mag[mag <= 10]
    
    r_aql_avg_mag.append(np.mean(mag))
    r_aql_dates.append(date_time)
print(r_aql_avg_mag)
print(r_aql_dates)

r_aql = pd.DataFrame({'Date': pd.to_datetime(r_aql_dates), 'Avg_Mag': r_aql_avg_mag})

standard_mag_files = sorted(glob('/raid6/users/rantoine/Mira_Notebooks/Better_Part_Three/*standard_mtab2.csv'))

standard_dates = []
standard_avg_mag = []
for filepath in standard_mag_files:
    filename = os.path.basename(filepath)
    date_str = filename.split('_standard')[0] #taking the date from the filename
    date_time = datetime.strptime(date_str, "%Y-%m-%d")
    #date = Time(date_time, scale="utc")
    
    df = pd.read_csv(filepath)
    mag = df['new_mag']
    mag = mag[mag <= 12]
    standard_avg_mag.append(np.mean(mag))
    standard_dates.append(date_time)
print(standard_avg_mag)
print(standard_dates)

standard = pd.DataFrame({'Date': pd.to_datetime(standard_dates), 'Avg_Mag': standard_avg_mag})

In [ ]:
plt.close()


plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
    'figure.titlesize': 18
})

fig, ax = plt.subplots(figsize=(10, 5))

ax.scatter(r_aql['Date'], r_aql['Avg_Mag'], color='#9C179E', s=10, label = 'R Aql')
ax.scatter(standard['Date'], standard['Avg_Mag'], color = 'b', s=10, label = 'Standard')
ax.set_xlabel('Observation')
ax.set_ylabel('New Magnitude')
ax.invert_yaxis()
ax.set_title('Yerkes Light Curve of R Aql and Standard Star')


plt.legend()
fig.autofmt_xdate()
plt.savefig('/raid6/users/rantoine/Mira_Notebooks/Better_Part_Three/something_is_wrong_light_curve.png')
plt.show()
